In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix ,classification_report
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [2]:
df = pd.read_csv('dementia_patients_health_data.csv')

In [3]:
most_frequent = df['Chronic_Health_Conditions'].mode()[0]
df['Chronic_Health_Conditions'].fillna(most_frequent, inplace = True)

C:\Users\91826\AppData\Local\Temp\ipykernel_2628\3794373468.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Chronic_Health_Conditions'].fillna(most_frequent, inplace = True)


In [4]:
df = df.drop(columns=['Prescription', 'Dosage in mg', 'Education_Level','MRI_Delay','Dominant_Hand'])

In [5]:
df.replace({
    "Family_History": {'No': 0, 'Yes': 1},
    "Gender": {'Male': 0, 'Female': 1},
    "Depression_Status": {'No': 0, 'Yes': 1},
    "Medication_History": {'No': 0, 'Yes': 1},
    "Sleep_Quality":{'Poor':0,'Good':1},
    "APOE_ε4": {'Negative': 0, 'Positive': 1},

    "Smoking_Status":{'Former Smoker':0,'Current Smoker':1,'Never Smoked':2},
    "Physical_Activity":{'Sedentary':0,'Moderate Activity':1,'Mild Activity':2},           
    "Nutrition_Diet":{'Low-Carb Diet':0,'Mediterranean Diet':1,'Balanced Diet':2},
    "Chronic_Health_Conditions":{'Diabetes':0,'Heart Disease':1,'Hypertension':2},
}, inplace = True)

C:\Users\91826\AppData\Local\Temp\ipykernel_2628\3508426344.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({


In [6]:
X = df.drop(columns = ['Dementia'])
y = df['Dementia']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [7]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)



scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


svm_model = SVC(kernel ='sigmoid', C = 1.0, gamma ='scale')  # You can adjust kernel, C, and gamma
svm_model.fit(X_train, y_train)


y_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cf=confusion_matrix(y_test,y_pred)
print(cf)
print(f"Accuracy: {accuracy*100}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")

[[ 94   0]
 [  4 102]]
Accuracy: 98.0
Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        94
           1       1.00      0.96      0.98       106

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200



In [15]:
import pandas as pd
import random
import pickle

# Define possible values for each feature
possible_values = {
    'Diabetic': [1, 0],
    'AlcoholLevel': [0, 1, 2, 3],
    'HeartRate': range(60, 101),
    'BloodOxygenLevel': range(90, 100),
    'BodyTemperature': [round(x * 0.1, 1) for x in range(360, 376)],
    'Weight': range(50, 101),
    'Age': range(20, 90),
    'Gender': [0, 1],  # 0 = Male, 1 = Female
    'Family_History': [1, 0],
    'Smoking_Status': [0, 1, 2],  # 0 = Former, 1 = Current, 2 = Never Smoked
    'APOE_ε4': [0, 1],  # 0 = Negative, 1 = Positive
    'Physical_Activity': [0, 1, 2],  # 0 = Sedentary, 1 = Moderate, 2 = Mild
    'Depression_Status': [0, 1],  # 0 = No, 1 = Yes
    'Cognitive_Test_Scores': range(20, 36),
    'Medication_History': [0, 1],  # 0 = No, 1 = Yes
    'Nutrition_Diet': [0, 1, 2],  # 0 = Low-Carb, 1 = Mediterranean, 2 = Balanced
    'Sleep_Quality': [0, 1],  # 0 = Poor, 1 = Good
    'Chronic_Health_Conditions': [0, 1, 2]  # 0 = Diabetes, 1 = Heart Disease, 2 = Hypertension
}

# Generate synthetic new patient data
def generate_new_patients(num_samples=5):
    data = []
    for _ in range(num_samples):
        patient = {feature: random.choice(possible_values[feature]) if isinstance(possible_values[feature], list) else random.randint(min(possible_values[feature]), max(possible_values[feature])) for feature in possible_values}
        data.append(patient)
    return pd.DataFrame(data)

# Generate new patient data
new_patients = generate_new_patients(num_samples=10)

# Load the trained SVM model
with open('svm_model.pkl', 'rb') as model_file:
    svm_model = pickle.load(model_file)

# Load the fitted scaler
with open('scaler.pkl', 'rb') as scaler_file:
    scaler = pickle.load(scaler_file)

# Scale features
new_patients_scaled = scaler.transform(new_patients)

# Predict whether patients have dementia
predictions = svm_model.predict(new_patients_scaled)

# Add predictions to the DataFrame
new_patients['Dementia'] = predictions



In [8]:
# Save the trained SVM model
with open('svm_model.pkl', 'wb') as model_file:
    pickle.dump(svm_model, model_file)

# Save the fitted scaler
with open('scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

In [9]:
feature = list(X.columns)
print(feature)

['Diabetic', 'AlcoholLevel', 'HeartRate', 'BloodOxygenLevel', 'BodyTemperature', 'Weight', 'Age', 'Gender', 'Family_History', 'Smoking_Status', 'APOE_ε4', 'Physical_Activity', 'Depression_Status', 'Cognitive_Test_Scores', 'Medication_History', 'Nutrition_Diet', 'Sleep_Quality', 'Chronic_Health_Conditions']


In [ ]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix ,classification_report
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

df = pd.read_csv('dementia_patients_health_data.csv')

most_frequent = df['Chronic_Health_Conditions'].mode()[0]
df['Chronic_Health_Conditions'].fillna(most_frequent, inplace = True)

df = df.drop(columns=['Prescription', 'Dosage in mg', 'Education_Level','MRI_Delay','Dominant_Hand'])

df.replace({
    "Family_History": {'No': 0, 'Yes': 1},
    "Gender": {'Male': 0, 'Female': 1},
    "Depression_Status": {'No': 0, 'Yes': 1},
    "Medication_History": {'No': 0, 'Yes': 1},
    "Sleep_Quality":{'Poor':0,'Good':1},
    "APOE_ε4": {'Negative': 0, 'Positive': 1},

    "Smoking_Status":{'Former Smoker':0,'Current Smoker':1,'Never Smoked':2},
    "Physical_Activity":{'Sedentary':0,'Moderate Activity':1,'Mild Activity':2},           
    "Nutrition_Diet":{'Low-Carb Diet':0,'Mediterranean Diet':1,'Balanced Diet':2},
    "Chronic_Health_Conditions":{'Diabetes':0,'Heart Disease':1,'Hypertension':2},
}, inplace = True)

X = df.drop(columns = ['Dementia'])
y = df['Dementia']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)



scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


svm_model = SVC(kernel ='sigmoid', C = 1.0, gamma ='scale')  # You can adjust kernel, C, and gamma
svm_model.fit(X_train, y_train)


y_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cf=confusion_matrix(y_test,y_pred)
print(cf)
print(f"Accuracy: {accuracy*100}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")